## Create variable about number of conflicts in previous month

* input:
    + `acs_conflict.csv` for number of events
    + `joined_ACLED_ZIP_22_PA.csv` for event id's
* process:
    + restructure datasets and select needed variables
    + create `df_merged_less`: merge df with num of conflicts onto df with event id's
    + create `df_merged_more`: merge df with event id's onto df with num of conflicts
* output:
    + `df_merged_less`:
        + contains only obs per event id
        + very few obs (761)
    + `df_merged_more`:
        + contains all obs per zip code and month (22481)
        + many NAs for event id's
        + binary var created for whether conflict present in previous month in same zip code

<br>
<hr>

### 0. Setup

In [ ]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

In [1]:
# import necessary packages
import numpy as np
import pandas as pd
import os

<br>
<hr>
<br>

### 1. Create variable for conflict in previous month

In [2]:
# load conflict data
df_events = pd.read_csv("./Dataset/create_acs_conflict/acs_conflict.csv")

# remove unnecessary vars
df_events_small = df_events[['ZCTA', 'Month', 'event_count']].copy()

# sort df by zip code and month
df_events_small = df_events_small.sort_values(by = ['ZCTA', 'Month'])

# create `event_count_prev` for number of events in previous month by zip code
df_events_small['event_count_prev'] = df_events_small.groupby('ZCTA')['event_count'].shift()

# `event_count_prev` in first month of each zip code set to NA
df_events_small.loc[df_events_small['Month'] == '2022-01', 'event_count_prev'] = pd.NA

# peek into df
print(df_events_small.head())

      ZCTA    Month  event_count  event_count_prev
0  15001.0  2022-01          1.0               NaN
1  15001.0  2022-02          0.0               1.0
2  15001.0  2022-03          0.0               0.0
3  15001.0  2022-04          0.0               0.0
4  15001.0  2022-05          0.0               0.0


<hr>
<br>

### 2. Prepare ACLED df for merging

In [3]:
# load acled data
ACLED_df = pd.read_csv('./1_Features/a_Spatial_Joins/2022/joined_ACLED_ZIP_22_PA.csv')

# remove unnecessary vars
df_acled_small = ACLED_df[['event_id_cnty', 'event_date', 'ZCTA5CE10']].copy()

# convert date var to YYYY-MM format
df_acled_small.loc[:, 'parsed_date'] = pd.to_datetime(df_acled_small['event_date'], format = '%d %B %Y')
df_acled_small['Month'] = df_acled_small['parsed_date']
df_acled_small['Month'] = df_acled_small['Month'].dt.strftime('%Y-%m')

# prepare for merging
df_acled_small = df_acled_small[['event_id_cnty', 'Month', 'ZCTA5CE10']]
df_acled_small = df_acled_small.rename(columns = {'ZCTA5CE10': 'ZCTA'})

# peek into df
print(df_acled_small.head())

  event_id_cnty    Month   ZCTA
0      USA50610  2022-12  15219
1      USA50946  2022-12  15219
2      USA51011  2022-12  15219
3      USA51023  2022-12  15219
4      USA50945  2022-12  15219


<hr>
<br>

### 3. Check data types are compatible

In [4]:
# var types in `df_events_small`
print("Types of variables in df_events_small:")
print(df_events_small.dtypes)

# var types in `df_acled_small`
print("\nTypes of variables in df_acled_small:")
print(df_acled_small.dtypes)

Types of variables in df_events_small:
ZCTA                float64
Month                object
event_count         float64
event_count_prev    float64
dtype: object

Types of variables in df_acled_small:
event_id_cnty    object
Month            object
ZCTA              int64
dtype: object


<hr>

### 4. Merge ACLED and events datasets

à la two ways:
1. keep only zips / months with an `event_id_cnty`
2. keep all zips / months with many NAs in `event_id_cnty`

<hr>

#### 4.1 Merge events data onto ACLED data
* only observations with an `event_id_cnty` from the ACLED dataset are shown
* this means only 761 observations

In [5]:
# merge, but only show obs with an `event_id_cnty`
df_merged_less = df_acled_small.merge(df_events_small, on = ['Month', 'ZCTA'], how = 'left')

# sort df by zip code, month and event id
df_merged_less = df_merged_less.sort_values(by = ['ZCTA', 'Month', 'event_id_cnty'])

# peek into results
print(df_merged_less.shape)
print("\n", df_merged_less.head())

(761, 5)

     event_id_cnty    Month   ZCTA  event_count  event_count_prev
639      USA38494  2022-01  15001          1.0               NaN
638      USA47365  2022-09  15001          1.0               0.0
712      USA44151  2022-06  15009          2.0               0.0
711      USA44181  2022-06  15009          2.0               0.0
730      USA42467  2022-05  15010          1.0               0.0


<hr>
<br>

#### 4.2 Merge ACLED data onto events data
* all observations (12 months per zip code) are shown
* this means all 22481 observations are there, but many NAs in `event_id_cnty`

In [6]:
## 4.2 merge events and acled data

# merge, and shows all obs with many NaN in `event_id_cnty`
df_merged_more = df_events_small.merge(df_acled_small, on = ['ZCTA', 'Month'], how = 'left')

#------------------------------------------------------------------------------

## generate `event_id_cnty_prev` for event id in previous month by zip code

# sort df by zip code and month
df_merged_more = df_merged_more.sort_values(by = ['ZCTA', 'Month'])

# create `event_id_cnty_prev` for number of events in previous month by zip code
df_merged_more['event_id_cnty_prev'] = df_merged_more.groupby('ZCTA')['event_id_cnty'].shift()

# `event_id_cnty_prev` in first month of each zip code set to NA
df_merged_more.loc[df_merged_more['Month'] == '2022-01', 'event_id_cnty_prev'] = pd.NA


# peek into results
print(df_merged_more.shape)
print("\n", df_merged_more.head())


(22481, 6)

       ZCTA    Month  event_count  event_count_prev event_id_cnty  \
0  15001.0  2022-01          1.0               NaN      USA38494   
1  15001.0  2022-02          0.0               1.0           NaN   
2  15001.0  2022-03          0.0               0.0           NaN   
3  15001.0  2022-04          0.0               0.0           NaN   
4  15001.0  2022-05          0.0               0.0           NaN   

  event_id_cnty_prev  
0               <NA>  
1           USA38494  
2                NaN  
3                NaN  
4                NaN  


In [7]:
# create a binary variable for whether there was an event in the previous month
df_merged_more['event_prev_same_zip'] = (df_merged_more['event_count_prev'] > 0).astype(int)

# print df
print(df_merged_more.head())

      ZCTA    Month  event_count  event_count_prev event_id_cnty  \
0  15001.0  2022-01          1.0               NaN      USA38494   
1  15001.0  2022-02          0.0               1.0           NaN   
2  15001.0  2022-03          0.0               0.0           NaN   
3  15001.0  2022-04          0.0               0.0           NaN   
4  15001.0  2022-05          0.0               0.0           NaN   

  event_id_cnty_prev  event_prev_same_zip  
0               <NA>                    0  
1           USA38494                    1  
2                NaN                    0  
3                NaN                    0  
4                NaN                    0  


In [8]:
# look at zip codes with repeated conflicts
df_merged_more[(df_merged_more['event_count'] > 0) & (df_merged_more['event_count_prev'] > 0)]

,ZCTA,Month,event_count,event_count_prev,event_id_cnty,event_id_cnty_prev,event_prev_same_zip
1412,15219.0,2022-02,10.0,5.0,USA39734,USA37937,1
1413,15219.0,2022-02,10.0,5.0,USA39415,USA39734,1
1414,15219.0,2022-02,10.0,5.0,USA39416,USA39415,1
1415,15219.0,2022-02,10.0,5.0,USA39417,USA39416,1
1416,15219.0,2022-02,10.0,5.0,USA39418,USA39417,1
...,...,...,...,...,...,...,...
21509,19406.0,2022-06,2.0,1.0,USA44490,USA45197,1
21735,19464.0,2022-04,1.0,1.0,USA42450,USA40723,1
22361,19601.0,2022-04,1.0,2.0,USA41996,USA40757,1
22362,19601.0,2022-05,1.0,1.0,USA42470,USA41996,1


<hr>
<br>

#### 5. Save updated dataset
**when certain that changes are correct, edit the name of the `.csv` file below to overwrite the `acs_conflict.csv`

In [10]:
# order the 4.1 df
df = df_merged_less.sort_values(by = ['ZCTA', 'Month', 'event_id_cnty'])

# order the 4.2 df
#df = df_merged_more.sort_values(by = ['ZCTA', 'Month'])

# save df
df.to_csv("./Dataset/create_acs_conflict/acs_conflict_varv.csv", index = False)

# overwrite acs_conflict.csv
#df.to_csv("./Dataset/create_acs_conflict/acs_conflict.csv", index = False)